In [ ]:
!pip install sentence-transformers==5.6 transformers torch python-Levenshtein scipy scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.8 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [ ]:
import os
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"   # default is 10s; raise it
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # faster, more robust downloads

In [ ]:
# Hugging Face login - required for gated models (e.g. google/embeddinggemma-300m)
from huggingface_hub import login
login()  # paste your HF token when prompted

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


# Exam 1 - Measurement geometry

**Definitions** (functions, ranges, units, plotting).

Changes vs. the previous version:

* symmetric similarity matrices are drawn as the upper triangle only
  (`plot_matrix(..., triangle=True)`); the asymmetric *Digits <-> Words*
  column is still drawn in full;
* the conversion line plots get headroom above 1.0 so the legend cannot
  overlap the curves;
* `range_texts` / `range_labels` factored out, so Exam 3 builds exactly the
  same strings;
* the overview figure is saved once instead of once per row, and the
  per-pair debug prints in `alignment_curve` are gone.

Nothing that touches a reported number was changed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import gc
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.ndimage import gaussian_filter1d
from scipy.stats import kendalltau
import pandas as pd
import os

# Output directories
RESULTS_DIR = "produced_results"
CSV_DIR = os.path.join(RESULTS_DIR, "csv")
PDF_DIR = os.path.join(RESULTS_DIR, "pdf")
TEX_DIR = os.path.join(RESULTS_DIR, "tex")
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TEX_DIR, exist_ok=True)


# --------------------------------------------------------------------
# NUMERIC RANGES
# --------------------------------------------------------------------

RANGES = {
    "local_0_10": np.arange(0, 10.5, 0.5),
    "medium_0_1000": np.arange(0, 1001, 50),
    "log_1_100k": np.unique(np.round(np.logspace(0, 5, 25))).astype(int),

    # NEW (2) sign / unit confusion baseline
    "sign_split": np.arange(-100, 101, 10),

    # NEW (3) scientific notation scale test
    "scientific": np.logspace(-6, 6, 25),
}


# --------------------------------------------------------------------
# UNIT GROUPS (SOURCE OF TRUTH)
# --------------------------------------------------------------------

UNIT_GROUPS = {
    "length": [
        ("meter", "centimeter", lambda v: v * 100),
        ("meter", "millimeter", lambda v: v * 1000),
        ("meter", "kilometer", lambda v: v / 1000),
    ],
    "mass": [
        ("kilogram", "gram", lambda v: v * 1000),
        ("gram", "milligram", lambda v: v * 1000),
        ("kilogram", "ton", lambda v: v / 1000),
    ],
    "volume": [
        ("liter", "milliliter", lambda v: v * 1000),
        ("liter", "gallon", lambda v: v * 3.785),
    ],
    "time": [
        ("second", "millisecond", lambda v: v * 1000),
        ("minute", "second", lambda v: v * 60),
        ("hour", "minute", lambda v: v * 60),
    ],
}


# --------------------------------------------------------------------
# FORMATTING
# --------------------------------------------------------------------

def format_unit(v, unit):
    unit_form = unit if abs(v) == 1 else unit + "s"
    return f"{v:g} {unit_form}"


def range_texts(unit, range_name, values):
    """Strings fed to the embedding model for one (unit, range) cell."""
    if range_name == "scientific":
        return [f"{v:.0e} {unit}" for v in values]
    return [format_unit(v, unit) for v in values]


def range_labels(range_name, values):
    """Tick labels for one range."""
    if range_name == "scientific":
        return [f"{v:.0e}" for v in values]
    return [f"{v:g}" for v in values]


# --------------------------------------------------------------------
# BASIC SIMILARITIES
# --------------------------------------------------------------------

def numeric_similarity(unit, values, model):
    texts = [format_unit(v, unit) for v in values]
    emb = model.encode(texts, normalize_embeddings=True)
    return cosine_similarity(emb)


def digit_word_similarity(unit, digits, words, model):
    emb_digits = model.encode([f"{d} {unit}" for d in digits], normalize_embeddings=True)
    emb_words = model.encode([f"{w} {unit}" for w in words], normalize_embeddings=True)
    return cosine_similarity(emb_words, emb_digits)


# --------------------------------------------------------------------
# ALIGNMENT CURVE (CORRECT + CONSISTENT)
# --------------------------------------------------------------------

def alignment_curve(unit_a, unit_b, converter, model):

    values = np.logspace(-2, 3, 40)

    texts_a = [format_unit(v, unit_a) for v in values]
    texts_b = [format_unit(converter(v), unit_b) for v in values]

    emb_a = model.encode(texts_a, normalize_embeddings=True).astype(np.float32)
    emb_b = model.encode(texts_b, normalize_embeddings=True).astype(np.float32)

    # explicit normalization safety (IMPORTANT FIX)
    emb_a = emb_a / np.linalg.norm(emb_a, axis=1, keepdims=True)
    emb_b = emb_b / np.linalg.norm(emb_b, axis=1, keepdims=True)

    # cosine similarity per aligned pair
    curve = np.sum(emb_a * emb_b, axis=1)

    curve_plot = gaussian_filter1d(curve, sigma=0.8)

    return values, curve_plot


def ranking_metrics(anchor_emb, candidate_embs, true_values):
    """
    Measures whether embeddings preserve correct ordering
    w.r.t physical magnitude.
    """

    # cosine similarity to anchor
    sim = cosine_similarity([anchor_emb], candidate_embs)[0]

    # --- Kendall tau ---
    tau, _ = kendalltau(true_values, -sim)

    # --- Pairwise accuracy ---
    correct = 0
    total = 0

    for i in range(len(sim)):
        for j in range(i + 1, len(sim)):

            # true ordering
            if true_values[i] < true_values[j]:
                correct += sim[i] > sim[j]
            else:
                correct += sim[i] < sim[j]

            total += 1

    pairwise_acc = correct / total if total > 0 else 0.0

    return tau, pairwise_acc


# --------------------------------------------------------------------
# PLOT HELPERS
# --------------------------------------------------------------------

def mask_lower(sim):
    """Blank the strictly lower triangle of a symmetric similarity matrix."""
    data = np.asarray(sim, dtype=float)
    if data.ndim != 2 or data.shape[0] != data.shape[1]:
        return data
    data = data.copy()
    data[np.tril_indices_from(data, k=-1)] = np.nan
    return data


def _triangle_cmap():
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="white", alpha=0.0)
    return cmap


def plot_matrix(sim, xt, yt, title, ax, ticks=None, triangle=False):

    data = mask_lower(sim) if triangle else np.asarray(sim, dtype=float)

    im = ax.imshow(
        data,
        origin="lower",
        cmap=_triangle_cmap(),
        vmin=0,
        vmax=1,
        interpolation="nearest",
        aspect="equal",
    )

    if title:
        ax.set_title(title, fontsize=9)

    if ticks is None:
        ticks = np.arange(len(xt))

    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [xt[i] for i in ticks],
        rotation=90,
        fontsize=10,
    )

    ax.set_yticks(ticks)
    ax.set_yticklabels(
        [yt[i] for i in ticks],
        fontsize=10,
    )

    ax.tick_params(
        axis="both",
        which="both",
        length=2,
        width=0.5,
        pad=1,
    )

    return im


# --------------------------------------------------------------------
# SETUP
# --------------------------------------------------------------------

digits = [str(i) for i in range(11)] + ["100"]

words = [
    "zero","one","two","three","four",
    "five","six","seven","eight","nine","ten", "one-hundred"
]

units = ["meter", "kilogram", "liter", "second"]


# --------------------------------------------------------------------
# COMPUTE RESULTS
# --------------------------------------------------------------------
def run_benchmark(model_name, results):

    model_id = model_name.split("/")[-1]
    model_id = model_id.replace("-", "_")
    print("=" * 80)
    print(model_name)
    print("=" * 80)

    model = SentenceTransformer(
        model_name,
        trust_remote_code=True,
    ).float()

    numeric_results = {}
    digit_results = {}
    alignment_results = {}

    for unit in units:

        numeric_results[unit] = {}

        for name, values in RANGES.items():
            texts = range_texts(unit, name, values)
            emb = model.encode(texts, normalize_embeddings=True)

            # similarity matrix (your existing visualization stays unchanged)
            sim = cosine_similarity(emb)

            # ranking metrics (NEW)
            anchor_emb = emb[0]
            true_values = np.array(values)

            tau, pair_acc = ranking_metrics(anchor_emb, emb, true_values)

            numeric_results[unit][name] = (sim, values, tau, pair_acc)

        digit_results[unit] = digit_word_similarity(unit, digits, words, model)


    # alignment curves (ONLY ONE SOURCE OF TRUTH)
    alignment_results = {}

    for group, pairs in UNIT_GROUPS.items():

        alignment_results[group] = []

        for ua, ub, conv in pairs:

            vals, curve = alignment_curve(ua, ub, conv, model)

            alignment_results[group].append(
                (ua, ub, vals, curve)
            )
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(7.5, 6),
    )

    titles = {
        "length": "Length",
        "mass": "Mass",
        "volume": "Volume",
        "time": "Time",
    }

    axis_map = {
        "length": axes[0,0],
        "mass": axes[0,1],
        "volume": axes[1,0],
        "time": axes[1,1],
    }

    styles = ["-", "--", "-.", ":"]

    for group, curves in alignment_results.items():

        ax = axis_map[group]

        for i, (ua, ub, values, curve) in enumerate(curves):

            ax.plot(
                values,
                curve,
                linewidth=2,
                linestyle=styles[i],
                label=f"{ua} \u2194 {ub}",
            )

        ax.set_xscale("log")

        # headroom above 1.0 so the legend never overlaps the curves
        ax.set_ylim(0.4, 1.25)
        ax.set_yticks(np.arange(0.4, 1.01, 0.1))

        ax.set_title(titles[group], fontsize=11, fontweight="bold")

        ax.set_xlabel("Physical value")
        ax.set_ylabel("Cosine similarity")

        ax.grid(alpha=0.25)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(
            frameon=False,
            fontsize=8,
            loc="upper right",
            handlelength=1.6,
            labelspacing=0.25,
            borderaxespad=0.2,
        )

    plt.tight_layout()

    plt.savefig(
        os.path.join(PDF_DIR, f"{model_id}_alignment_overview.pdf"),
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

    # --------------------------------------------------------------------
    # INDIVIDUAL PDFs
    # --------------------------------------------------------------------

    for unit in units:

        for name, (sim, values, _, _) in numeric_results[unit].items():

            plt.figure(figsize=(6,5))
            plt.imshow(
                mask_lower(sim),
                origin="lower",
                cmap=_triangle_cmap(),
                vmin=0,
                vmax=1,
            )

            labels = range_labels(name, values)
            ticks = np.arange(len(values))

            plt.xticks(ticks, labels, rotation=90, fontsize=6)
            plt.yticks(ticks, labels, fontsize=6)

            plt.title(f"{unit} ({name})")
            plt.colorbar()

            plt.tight_layout()
            plt.savefig(
            os.path.join(PDF_DIR, f"{model_id}_{unit}_{name}.pdf"),
            bbox_inches="tight",
            )
            plt.close()

        plt.figure(figsize=(6,5))
        plt.imshow(digit_results[unit], origin="lower", cmap="viridis", vmin=0, vmax=1)

        plt.xticks(range(len(digits)), digits, rotation=90, fontsize=6)
        plt.yticks(range(len(words)), words, fontsize=6)

        plt.title(f"{unit} digits vs words")
        plt.colorbar()

        plt.tight_layout()
        plt.savefig(
        os.path.join(PDF_DIR, f"{model_id}_{unit}_digits_words.pdf"),
        bbox_inches="tight",
            )
        plt.close()


    # --------------------------------------------------------------------
    # OVERVIEW FIGURE
    # --------------------------------------------------------------------

    cols = len(RANGES) + 1
    range_names = [
    "local_0_10",
    "medium_0_1000",
    "log_1_100k",
    "sign_split",
    "scientific",
    ]


    column_titles = [
    "Local",
    "Medium",
    "Log",
    "Sign",
    "Scientific",
    "Digits \u2194 Words",
    ]

    import matplotlib.gridspec as gridspec

    fig = plt.figure(figsize=(2.2 * cols, 2.0 * len(units)))

    gs = gridspec.GridSpec(
        nrows=len(units),
        ncols=cols,
        figure=fig,
        width_ratios=[1, 1, 1, 1, 1.0, 2.0],  # <-- IMPORTANT CHANGE
        wspace=0.02,
        hspace=0.12,
    )

    axes = np.empty((len(units), cols), dtype=object)

    for r in range(len(units)):
        for c in range(cols):
            axes[r, c] = fig.add_subplot(gs[r, c])

    fig.subplots_adjust(
        left=0.11,
        right=0.995,
        top=0.92,
        bottom=0.08,
        wspace=0.02,
        hspace=0.08,
    )

    # --------------------------------------------------------
    # Column headers
    # --------------------------------------------------------

    for c, title in enumerate(column_titles):
        axes[0, c].set_title(
            title,
            fontsize=11,
            fontweight="bold",
            pad=6,
        )

    # --------------------------------------------------------
    # Rows
    # --------------------------------------------------------
    row_labels = {
    "meter": "meter",
    "kilogram": "kilogram",
    "liter": "liter",
    "second": "second",
    }
    for r, unit in enumerate(units):

        # row label
        axes[r, 0].set_ylabel(
            row_labels[unit],
            fontsize=11,
            fontweight="bold",
            rotation=0,
            labelpad=35,
            va="center",
        )

        # numeric benchmarks
        for c, name in enumerate(range_names):

            sim, values, _, _ = numeric_results[unit][name]
            if len(values) == 21:
                ticks = [0, 5, 10, 15, 20]
            elif len(values) == 25:
                ticks = [0, 6, 12, 18, 24]
            else:
                ticks = np.arange(len(values))

            labels = range_labels(name, values)

            plot_matrix(
                sim,
                labels,
                labels,
                "",
                axes[r, c],
                ticks=ticks,
                triangle=True,   # symmetric -> upper triangle only
            )

        # digits <-> words (asymmetric -> full matrix)
        plot_matrix(
            digit_results[unit],
            [str(d) for d in digits],
            words,
            "",
            axes[r, len(range_names)],
        )

    # --------------------------------------------------------
    # Remove redundant tick labels
    # --------------------------------------------------------

    for r in range(len(units)):
        for c in range(cols):

            ax = axes[r, c]

            # digits column: keep labels
            if c == len(range_names):
                continue

            # only bottom row keeps x labels
            if r != len(units) - 1:
                ax.set_xticklabels([])

            # only first column keeps y labels
            #if c != 0:
            ax.set_yticklabels([])

    plt.savefig(
        os.path.join(PDF_DIR, f"{model_id}_embedding_overview.pdf"),
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

    print(f"\n=== {model_id} ===")

    for unit in units:
        print(f"\n[{unit}]")

        for name in RANGES.keys():
            _, _, tau, acc = numeric_results[unit][name]
            print(f"{name:15s}  \u03c4={tau:.3f}   pairwise={acc:.3f}")

            results.append({
                "model": model_id,
                "unit": unit,
                "range": name,
                "kendall": tau,
                "pairwise": acc,
            })

    df = pd.DataFrame(results)
    summary = df.groupby("model").agg(
    kendall_mean=("kendall", "mean"),
    pairwise_mean=("pairwise", "mean"),
    kendall_std=("kendall", "std"),
    pairwise_std=("pairwise", "std"),
    ).reset_index()
    df.to_csv(os.path.join(CSV_DIR, f"{model_id}_benchmark_full.csv"), index=False)
    summary.to_csv(os.path.join(CSV_DIR, f"{model_id}_benchmark_summary.csv"), index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()

def run_measurement(models):
    """Run the measurement benchmark over a list of model names."""
    results = []
    for model_name in models:
        run_benchmark(model_name, results)
    return results

### Run: baseline models

In [ ]:
# Baseline models (the original paper set + Qwen3 4B / 8B)
MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "intfloat/e5-large-v2",
    "BAAI/bge-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
    "Qwen/Qwen3-Embedding-0.6B",
    "Qwen/Qwen3-Embedding-4B",
    "Qwen/Qwen3-Embedding-8B",
]

run_measurement(MODELS)

sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]


=== Qwen3_Embedding_8B ===

[meter]
local_0_10       τ=0.029   pairwise=0.514
medium_0_1000    τ=0.286   pairwise=0.643
log_1_100k       τ=0.593   pairwise=0.797
sign_split       τ=0.200   pairwise=0.600
scientific       τ=0.533   pairwise=0.767

[kilogram]
local_0_10       τ=-0.210   pairwise=0.395
medium_0_1000    τ=-0.019   pairwise=0.490
log_1_100k       τ=0.540   pairwise=0.770
sign_split       τ=0.419   pairwise=0.710
scientific       τ=0.487   pairwise=0.743

[liter]
local_0_10       τ=-0.105   pairwise=0.448
medium_0_1000    τ=0.105   pairwise=0.552
log_1_100k       τ=0.613   pairwise=0.807
sign_split       τ=0.410   pairwise=0.705
scientific       τ=0.553   pairwise=0.777

[second]
local_0_10       τ=0.248   pairwise=0.624
medium_0_1000    τ=0.381   pairwise=0.690
log_1_100k       τ=0.407   pairwise=0.703
sign_split       τ=0.390   pairwise=0.695
scientific       τ=0.593   pairwise=0.797


[{'model': 'all_MiniLM_L6_v2',
  'unit': 'meter',
  'range': 'local_0_10',
  'kendall': np.float64(0.4571428571428572),
  'pairwise': np.float64(0.7285714285714285)},
 {'model': 'all_MiniLM_L6_v2',
  'unit': 'meter',
  'range': 'medium_0_1000',
  'kendall': np.float64(-0.13333333333333333),
  'pairwise': np.float64(0.43333333333333335)},
 {'model': 'all_MiniLM_L6_v2',
  'unit': 'meter',
  'range': 'log_1_100k',
  'kendall': np.float64(0.4866666666666665),
  'pairwise': np.float64(0.7433333333333333)},
 {'model': 'all_MiniLM_L6_v2',
  'unit': 'meter',
  'range': 'sign_split',
  'kendall': np.float64(0.5047619047619049),
  'pairwise': np.float64(0.7523809523809524)},
 {'model': 'all_MiniLM_L6_v2',
  'unit': 'meter',
  'range': 'scientific',
  'kendall': np.float64(0.39333333333333326),
  'pairwise': np.float64(0.6966666666666667)},
 {'model': 'all_MiniLM_L6_v2',
  'unit': 'kilogram',
  'range': 'local_0_10',
  'kendall': np.float64(0.29523809523809524),
  'pairwise': np.float64(0.6476190

### Run: extended models

Add model names to `NEW_MODELS` to extend the paper.

#### PT1

In [ ]:
# Extended run: add model names here to extend the paper.
NEW_MODELS_1 = [
    "sentence-transformers/LaBSE",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "intfloat/multilingual-e5-large-instruct",
    "BAAI/bge-m3",
    "google/embeddinggemma-300m",
]

run_measurement(NEW_MODELS_1)

sentence-transformers/LaBSE


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            


=== embeddinggemma_300m ===

[meter]
local_0_10       τ=0.133   pairwise=0.567
medium_0_1000    τ=0.162   pairwise=0.581
log_1_100k       τ=0.607   pairwise=0.803
sign_split       τ=0.143   pairwise=0.571
scientific       τ=0.433   pairwise=0.717

[kilogram]
local_0_10       τ=0.152   pairwise=0.576
medium_0_1000    τ=0.000   pairwise=0.500
log_1_100k       τ=0.467   pairwise=0.733
sign_split       τ=0.067   pairwise=0.533
scientific       τ=0.433   pairwise=0.717

[liter]
local_0_10       τ=0.143   pairwise=0.571
medium_0_1000    τ=0.048   pairwise=0.524
log_1_100k       τ=0.467   pairwise=0.733
sign_split       τ=0.029   pairwise=0.514
scientific       τ=0.393   pairwise=0.697

[second]
local_0_10       τ=0.171   pairwise=0.586
medium_0_1000    τ=0.048   pairwise=0.524
log_1_100k       τ=0.620   pairwise=0.810
sign_split       τ=0.086   pairwise=0.543
scientific       τ=0.280   pairwise=0.640


[{'model': 'LaBSE',
  'unit': 'meter',
  'range': 'local_0_10',
  'kendall': np.float64(0.39047619047619053),
  'pairwise': np.float64(0.6952380952380952)},
 {'model': 'LaBSE',
  'unit': 'meter',
  'range': 'medium_0_1000',
  'kendall': np.float64(0.5904761904761905),
  'pairwise': np.float64(0.7952380952380952)},
 {'model': 'LaBSE',
  'unit': 'meter',
  'range': 'log_1_100k',
  'kendall': np.float64(0.8466666666666665),
  'pairwise': np.float64(0.9233333333333333)},
 {'model': 'LaBSE',
  'unit': 'meter',
  'range': 'sign_split',
  'kendall': np.float64(0.4285714285714286),
  'pairwise': np.float64(0.7142857142857143)},
 {'model': 'LaBSE',
  'unit': 'meter',
  'range': 'scientific',
  'kendall': np.float64(0.4266666666666666),
  'pairwise': np.float64(0.7133333333333334)},
 {'model': 'LaBSE',
  'unit': 'kilogram',
  'range': 'local_0_10',
  'kendall': np.float64(0.40952380952380957),
  'pairwise': np.float64(0.7047619047619048)},
 {'model': 'LaBSE',
  'unit': 'kilogram',
  'range': 'me

#### PT2

In [ ]:
NEW_MODELS_2 = [
    "ibm-granite/granite-embedding-107m-multilingual",
    "ibm-granite/granite-embedding-278m-multilingual",
    "ibm-granite/granite-embedding-english-r2",
    "ibm-granite/granite-embedding-97m-multilingual-r2",
    "ibm-granite/granite-embedding-311m-multilingual-r2",
]

run_measurement(NEW_MODELS_2)

ibm-granite/granite-embedding-107m-multilingual


modules.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/611k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]


=== granite_embedding_311m_multilingual_r2 ===

[meter]
local_0_10       τ=0.486   pairwise=0.743
medium_0_1000    τ=0.495   pairwise=0.748
log_1_100k       τ=0.567   pairwise=0.783
sign_split       τ=0.029   pairwise=0.514
scientific       τ=0.333   pairwise=0.667

[kilogram]
local_0_10       τ=0.429   pairwise=0.714
medium_0_1000    τ=0.324   pairwise=0.662
log_1_100k       τ=0.400   pairwise=0.700
sign_split       τ=0.171   pairwise=0.586
scientific       τ=0.253   pairwise=0.627

[liter]
local_0_10       τ=0.429   pairwise=0.714
medium_0_1000    τ=0.362   pairwise=0.681
log_1_100k       τ=0.600   pairwise=0.800
sign_split       τ=0.219   pairwise=0.610
scientific       τ=0.287   pairwise=0.643

[second]
local_0_10       τ=0.248   pairwise=0.624
medium_0_1000    τ=0.200   pairwise=0.600
log_1_100k       τ=0.267   pairwise=0.633
sign_split       τ=0.314   pairwise=0.657
scientific       τ=0.333   pairwise=0.667


[{'model': 'granite_embedding_107m_multilingual',
  'unit': 'meter',
  'range': 'local_0_10',
  'kendall': np.float64(0.4761904761904762),
  'pairwise': np.float64(0.7380952380952381)},
 {'model': 'granite_embedding_107m_multilingual',
  'unit': 'meter',
  'range': 'medium_0_1000',
  'kendall': np.float64(0.13333333333333333),
  'pairwise': np.float64(0.5666666666666667)},
 {'model': 'granite_embedding_107m_multilingual',
  'unit': 'meter',
  'range': 'log_1_100k',
  'kendall': np.float64(0.7066666666666666),
  'pairwise': np.float64(0.8533333333333334)},
 {'model': 'granite_embedding_107m_multilingual',
  'unit': 'meter',
  'range': 'sign_split',
  'kendall': np.float64(-0.18095238095238098),
  'pairwise': np.float64(0.4095238095238095)},
 {'model': 'granite_embedding_107m_multilingual',
  'unit': 'meter',
  'range': 'scientific',
  'kendall': np.float64(0.18666666666666662),
  'pairwise': np.float64(0.5933333333333334)},
 {'model': 'granite_embedding_107m_multilingual',
  'unit': 'ki

#### PT3

In [ ]:
NEW_MODELS_3 = [
    "mixedbread-ai/mxbai-embed-large-v1",
    "microsoft/harrier-oss-v1-270m",
    "microsoft/harrier-oss-v1-0.6b",
    "lightonai/DenseOn",
]

run_measurement(NEW_MODELS_3)

mixedbread-ai/mxbai-embed-large-v1


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/114k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]


=== DenseOn ===

[meter]
local_0_10       τ=0.257   pairwise=0.629
medium_0_1000    τ=0.181   pairwise=0.590
log_1_100k       τ=0.713   pairwise=0.857
sign_split       τ=0.400   pairwise=0.700
scientific       τ=0.407   pairwise=0.703

[kilogram]
local_0_10       τ=0.219   pairwise=0.610
medium_0_1000    τ=0.305   pairwise=0.652
log_1_100k       τ=0.580   pairwise=0.790
sign_split       τ=0.638   pairwise=0.819
scientific       τ=0.287   pairwise=0.643

[liter]
local_0_10       τ=0.362   pairwise=0.681
medium_0_1000    τ=0.457   pairwise=0.729
log_1_100k       τ=0.640   pairwise=0.820
sign_split       τ=0.524   pairwise=0.762
scientific       τ=0.447   pairwise=0.723

[second]
local_0_10       τ=0.133   pairwise=0.567
medium_0_1000    τ=0.448   pairwise=0.724
log_1_100k       τ=0.567   pairwise=0.783
sign_split       τ=0.457   pairwise=0.729
scientific       τ=0.520   pairwise=0.760


[{'model': 'mxbai_embed_large_v1',
  'unit': 'meter',
  'range': 'local_0_10',
  'kendall': np.float64(0.4190476190476191),
  'pairwise': np.float64(0.7095238095238096)},
 {'model': 'mxbai_embed_large_v1',
  'unit': 'meter',
  'range': 'medium_0_1000',
  'kendall': np.float64(0.4952380952380953),
  'pairwise': np.float64(0.7476190476190476)},
 {'model': 'mxbai_embed_large_v1',
  'unit': 'meter',
  'range': 'log_1_100k',
  'kendall': np.float64(0.6199999999999999),
  'pairwise': np.float64(0.81)},
 {'model': 'mxbai_embed_large_v1',
  'unit': 'meter',
  'range': 'sign_split',
  'kendall': np.float64(0.29523809523809524),
  'pairwise': np.float64(0.6476190476190476)},
 {'model': 'mxbai_embed_large_v1',
  'unit': 'meter',
  'range': 'scientific',
  'kendall': np.float64(0.20666666666666664),
  'pairwise': np.float64(0.6033333333333334)},
 {'model': 'mxbai_embed_large_v1',
  'unit': 'kilogram',
  'range': 'local_0_10',
  'kendall': np.float64(0.4190476190476191),
  'pairwise': np.float64(0.

# Exam 2 - Lexical vs. numerical similarity

**Definitions** (number generation, distances, correlations).

Changes:

* the association statistic is now Kendall's tau (scipy's tau-b, which corrects
  for the heavy ties in the Levenshtein distances) instead of Spearman's rho,
  matching the caption of Table 3 and the statistic used in Tables 1, 2 and 4.
  **Every number in Table 3 changes** - tau-b is systematically smaller in
  magnitude than rho, so old and new values must not be mixed;
* results are written per model to `csv/<model>_lexical.csv` in long format, so
  the table can be rebuilt for any subset of models. Previously each
  `run_lexical` call overwrote the same `.tex`, which made the appendix version
  impossible;
* the LaTeX emission moved to the Tables section (`table_lexical`), where the
  other three tables are built. The old inline version also closed the tabular
  inside the model loop, so it only ever emitted one row.

In [ ]:
#!/usr/bin/env python3

"""
Investigate whether embedding similarity reflects numerical similarity.

For each embedding model:

- generate 50 floats + 50 ints
- embed all numbers as strings
- compute cosine similarity for every pair
- compare against
    * character Levenshtein
    * tokenizer Levenshtein
    * numerical distance

Reports Kendall's tau (tau-b, so ties in the Levenshtein distances are handled)
overall and for several subsets. Results are written per model to
    produced_results/csv/<model>_lexical.csv
and the LaTeX table is built from those files in the Tables section.
"""

from __future__ import annotations

import itertools
import random
import gc

import numpy as np
import pandas as pd
import torch
from scipy.stats import kendalltau
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from Levenshtein import distance as levenshtein
from tqdm import tqdm
import os

RESULTS_DIR = "produced_results"
CSV_DIR = os.path.join(RESULTS_DIR, "csv")
os.makedirs(CSV_DIR, exist_ok=True)

# ----------------------------------------------------------------------

SEED = 12345

N_FLOATS = 50
N_INTS = 50

LOW = -1000
HIGH = 1000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------


def run_lexical(models):
    random.seed(SEED)
    np.random.seed(SEED)


    def make_numbers():

        floats = [
            round(random.uniform(LOW, HIGH), 5)
            for _ in range(N_FLOATS)
        ]

        ints = [
            random.randint(LOW, HIGH)
            for _ in range(N_INTS)
        ]

        return floats, ints


    floats, ints = make_numbers()

    items = []

    for x in floats:
        items.append(
            {
                "value": float(x),
                "text": f"{x:.5f}",
                "type": "float",
                "sign": "positive" if x >= 0 else "negative",
            }
        )

    for x in ints:
        items.append(
            {
                "value": float(x),
                "text": str(x),
                "type": "int",
                "sign": "positive" if x >= 0 else "negative",
            }
        )


    # ----------------------------------------------------------------------


    def token_levenshtein(tokenizer, a: str, b: str):
        ta = tokenizer.encode(a, add_special_tokens=False)
        tb = tokenizer.encode(b, add_special_tokens=False)
        return levenshtein(tuple(ta), tuple(tb))


    def cosine_matrix(x):
        x = x / np.linalg.norm(x, axis=1, keepdims=True)
        return x @ x.T


    # ----------------------------------------------------------------------

    pair_records = []

    for i, j in itertools.combinations(range(len(items)), 2):

        A = items[i]
        B = items[j]

        pair_records.append(
            {
                "i": i,
                "j": j,
                "char_lev": levenshtein(A["text"], B["text"]),
                "num_dist": abs(A["value"] - B["value"]),
                "same_type": A["type"] == B["type"],
                "type_pair": f"{A['type']}-{B['type']}",
                "same_sign": A["sign"] == B["sign"],
                "sign_pair": f"{A['sign']}-{B['sign']}",
            }
        )

    pairs_df = pd.DataFrame(pair_records)

    # ----------------------------------------------------------------------


    def correlations(df):
        """Kendall's tau-b between embedding similarity and each reference."""

        sim = df["similarity"]

        return {
            "char_lev":
                kendalltau(sim, -df["char_lev"]).statistic,
            "token_lev":
                kendalltau(sim, -df["token_lev"]).statistic,
            "num_dist":
                kendalltau(sim, -df["num_dist"]).statistic,
        }

    # ----------------------------------------------------------------------

    results = {}

    for model_name in models:

        print("=" * 80)
        print(model_name)

        model_id = model_name.split("/")[-1].replace("-", "_")

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
        )

        token_dists = []

        for rec in tqdm(pair_records, leave=False, desc="token distances"):
            token_dists.append(
                token_levenshtein(
                    tokenizer,
                    items[rec["i"]]["text"],
                    items[rec["j"]]["text"],
                )
            )

        model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=DEVICE,
        ).float()

        embeddings = model.encode(
            [x["text"] for x in items],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )

        sim = cosine_matrix(embeddings)

        similarities = []

        for rec in pair_records:
            similarities.append(sim[rec["i"], rec["j"]])

        df = pairs_df.copy()
        df["similarity"] = similarities
        df["token_lev"] = token_dists

        # ------------------------------------------------------------------
        # every subset, in long format
        # ------------------------------------------------------------------

        rows = []

        def add(group_type, group, sub):
            for reference, tau in correlations(sub).items():
                rows.append({
                    "model": model_id,
                    "group_type": group_type,
                    "group": str(group),
                    "reference": reference,
                    "tau": tau,
                    "n_pairs": len(sub),
                })

        add("overall", "all", df)

        for group_type in ["type_pair", "same_sign", "sign_pair"]:
            for key, sub in df.groupby(group_type):
                add(group_type, key, sub)

        out = pd.DataFrame(rows)
        out.to_csv(os.path.join(CSV_DIR, f"{model_id}_lexical.csv"), index=False)

        print("\nKendall's tau (x100)")
        print(
            out[out.group_type.isin(["overall", "type_pair"])]
            .pivot(index="reference", columns="group", values="tau")
            .mul(100)
            .round(1)
            .to_string()
        )
        print()

        results[model_id] = out

        del model, tokenizer, embeddings, sim
        gc.collect()
        torch.cuda.empty_cache()

    print("Wrote per-model CSVs. Build the LaTeX table with table_lexical().")
    return results

### Run: baseline models

In [ ]:
# Baseline models (the original paper set + Qwen3 4B / 8B)
MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "intfloat/e5-large-v2",
    "BAAI/bge-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
    "Qwen/Qwen3-Embedding-0.6B",
    "Qwen/Qwen3-Embedding-4B",
    "Qwen/Qwen3-Embedding-8B",
]

run_lexical(MODELS)

sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   21.0         22.0       17.1     44.2
num_dist   28.1         23.0       28.7     39.0
token_lev  15.1         10.4       10.2     37.1

sentence-transformers/all-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   30.1         26.3       38.8     53.8
num_dist   13.1         12.6        8.4     27.3
token_lev  30.1         26.3       38.8     53.8

Qwen/Qwen3-Embedding-8B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   36.7         33.9       39.2     49.5
num_dist   18.3         24.2       15.8     25.1
token_lev  36.7         33.9       39.2     49.5

Wrote per-model CSVs. Build the LaTeX table with table_lexical().


{'all_MiniLM_L6_v2':                model group_type              group  reference       tau  \
 0   all_MiniLM_L6_v2    overall                all   char_lev  0.209675   
 1   all_MiniLM_L6_v2    overall                all  token_lev  0.150933   
 2   all_MiniLM_L6_v2    overall                all   num_dist  0.280550   
 3   all_MiniLM_L6_v2  type_pair        float-float   char_lev  0.220345   
 4   all_MiniLM_L6_v2  type_pair        float-float  token_lev  0.104305   
 5   all_MiniLM_L6_v2  type_pair        float-float   num_dist  0.230249   
 6   all_MiniLM_L6_v2  type_pair          float-int   char_lev  0.171122   
 7   all_MiniLM_L6_v2  type_pair          float-int  token_lev  0.101823   
 8   all_MiniLM_L6_v2  type_pair          float-int   num_dist  0.287474   
 9   all_MiniLM_L6_v2  type_pair            int-int   char_lev  0.442376   
 10  all_MiniLM_L6_v2  type_pair            int-int  token_lev  0.370988   
 11  all_MiniLM_L6_v2  type_pair            int-int   num_dist  0.39

### Run: extended models

Add model names to `NEW_MODELS` to extend the paper.

#### PT1

In [ ]:
# Extended run: add model names here to extend the paper.
NEW_MODELS_1 = [
    "sentence-transformers/LaBSE",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "intfloat/multilingual-e5-large-instruct",
    "BAAI/bge-m3",
    "google/embeddinggemma-300m",
]

run_lexical(NEW_MODELS_1)

sentence-transformers/LaBSE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   39.6         32.7       28.5     48.7
num_dist   18.1         29.7       18.8     24.4
token_lev  42.7         42.0       13.0     35.7

sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   18.1         31.3       36.8     49.2
num_dist   19.5         34.3       22.7     29.7
token_lev   3.6         25.0       25.7     31.9

google/embeddinggemma-300m


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   51.6         36.3       39.6     44.6
num_dist    3.3          8.3        2.1      8.8
token_lev  51.6         36.3       39.6     44.6

Wrote per-model CSVs. Build the LaTeX table with table_lexical().


{'LaBSE':     model group_type              group  reference       tau  n_pairs
 0   LaBSE    overall                all   char_lev  0.395809     4950
 1   LaBSE    overall                all  token_lev  0.427459     4950
 2   LaBSE    overall                all   num_dist  0.180788     4950
 3   LaBSE  type_pair        float-float   char_lev  0.327291     1225
 4   LaBSE  type_pair        float-float  token_lev  0.420275     1225
 5   LaBSE  type_pair        float-float   num_dist  0.297079     1225
 6   LaBSE  type_pair          float-int   char_lev  0.285077     2500
 7   LaBSE  type_pair          float-int  token_lev  0.130356     2500
 8   LaBSE  type_pair          float-int   num_dist  0.188060     2500
 9   LaBSE  type_pair            int-int   char_lev  0.486978     1225
 10  LaBSE  type_pair            int-int  token_lev  0.357097     1225
 11  LaBSE  type_pair            int-int   num_dist  0.244031     1225
 12  LaBSE  same_sign              False   char_lev  0.391798     24

#### PT2

In [ ]:
NEW_MODELS_2 = [
    "ibm-granite/granite-embedding-107m-multilingual",
    "ibm-granite/granite-embedding-278m-multilingual",
    "ibm-granite/granite-embedding-english-r2",
    "ibm-granite/granite-embedding-97m-multilingual-r2",
    "ibm-granite/granite-embedding-311m-multilingual-r2",
]

run_lexical(NEW_MODELS_2)

ibm-granite/granite-embedding-107m-multilingual


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   27.0         25.9       32.4     43.1
num_dist   16.7         14.9       16.4     20.6
token_lev  16.7         10.9       14.1     31.0

ibm-granite/granite-embedding-278m-multilingual


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev    4.9         17.5       11.9     27.9
num_dist   12.5         12.5       12.7     21.2
token_lev   8.5         18.9        2.5     29.3

ibm-granite/granite-embedding-311m-multilingual-r2


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   14.2         32.5       33.3     53.6
num_dist    9.1         17.0       10.4     11.8
token_lev  14.2         32.5       33.3     53.6

Wrote per-model CSVs. Build the LaTeX table with table_lexical().


{'granite_embedding_107m_multilingual':                                   model group_type              group  \
 0   granite_embedding_107m_multilingual    overall                all   
 1   granite_embedding_107m_multilingual    overall                all   
 2   granite_embedding_107m_multilingual    overall                all   
 3   granite_embedding_107m_multilingual  type_pair        float-float   
 4   granite_embedding_107m_multilingual  type_pair        float-float   
 5   granite_embedding_107m_multilingual  type_pair        float-float   
 6   granite_embedding_107m_multilingual  type_pair          float-int   
 7   granite_embedding_107m_multilingual  type_pair          float-int   
 8   granite_embedding_107m_multilingual  type_pair          float-int   
 9   granite_embedding_107m_multilingual  type_pair            int-int   
 10  granite_embedding_107m_multilingual  type_pair            int-int   
 11  granite_embedding_107m_multilingual  type_pair            int-int   

#### PT3

In [ ]:
NEW_MODELS_3 = [
    "mixedbread-ai/mxbai-embed-large-v1",
    "microsoft/harrier-oss-v1-270m",
    "microsoft/harrier-oss-v1-0.6b",
    "lightonai/DenseOn"
]

run_lexical(NEW_MODELS_3)

mixedbread-ai/mxbai-embed-large-v1


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}



Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   10.0         24.2       28.7     33.9
num_dist    6.9         18.0        6.1     16.5
token_lev   5.4         16.0       26.6     17.1

microsoft/harrier-oss-v1-270m


...


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   27.0         42.1       32.0     43.0
num_dist   21.7         46.1       29.7     32.6
token_lev  27.0         42.1       32.0     43.0

lightonai/DenseOn


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Kendall's tau (x100)
group       all  float-float  float-int  int-int
reference                                       
char_lev   23.5         18.1       21.0     36.9
num_dist   20.2         20.2       18.9     33.3
token_lev  27.4         17.0       25.1     31.6

Wrote per-model CSVs. Build the LaTeX table with table_lexical().


{'mxbai_embed_large_v1':                    model group_type              group  reference       tau  \
 0   mxbai_embed_large_v1    overall                all   char_lev  0.099826   
 1   mxbai_embed_large_v1    overall                all  token_lev  0.054175   
 2   mxbai_embed_large_v1    overall                all   num_dist  0.068817   
 3   mxbai_embed_large_v1  type_pair        float-float   char_lev  0.241846   
 4   mxbai_embed_large_v1  type_pair        float-float  token_lev  0.159513   
 5   mxbai_embed_large_v1  type_pair        float-float   num_dist  0.180180   
 6   mxbai_embed_large_v1  type_pair          float-int   char_lev  0.287213   
 7   mxbai_embed_large_v1  type_pair          float-int  token_lev  0.266303   
 8   mxbai_embed_large_v1  type_pair          float-int   num_dist  0.060714   
 9   mxbai_embed_large_v1  type_pair            int-int   char_lev  0.339039   
 10  mxbai_embed_large_v1  type_pair            int-int  token_lev  0.170679   
 11  mxbai_embed

# Exam 3 - Is similarity just miscalibrated?

**RQ.** Cosine similarity is a fixed, unweighted read-out of the embedding. If
the physical ordering *is* encoded but on directions cosine happens to ignore,
a learned read-out should recover it. If even a learned read-out fails, the
information is not there.

**Setup.** Instead of the dot product, fit a linear regressor on the absolute
embedding difference of a pair:

    d_hat(a, b) = w . |E(a) - E(b)| + b0

trained to predict `log1p(|v_a - v_b|)`. Data is split at the **item** level, so
no test pair shares a string with any training pair. One probe per
(model, range), pooled over the four quantities.

**Evaluation.** Every test item serves once as anchor; Kendall's tau and pairwise
accuracy are computed over the remaining test items and averaged. The whole
split-fit-evaluate loop is repeated over `REG_N_SPLITS = 5` random splits and
averaged, since a single split leaves only 7-9 test items. The identical
protocol is applied to plain cosine similarity, giving the directly comparable
`kendall_cosine` / `pairwise_cosine` columns. Note that, unlike Table 1, the
anchor is excluded from its own candidate list, so the trivially-resolved
diagonal inflates neither column.

Run the Exam 1 definitions cell first - this reuses `RANGES`, `units`,
`range_texts` and `CSV_DIR` from it.

**Output.** For each model, two files next to the existing ones:

    <model>_benchmark_full_regression_experiment.csv
    <model>_benchmark_summary_regression_experiment.csv

In [ ]:
"""
Exam 3 -- is embedding similarity just miscalibrated for measurement?

Instead of reading the physical ordering off the cosine similarity, we fit a
linear regressor on the *absolute embedding difference* of a pair and let it
predict the physical distance of that pair.

    d_hat(a, b) = w . |E(a) - E(b)| + b0

If a linear read-out recovers the physical ordering, the information is present
in the embedding and cosine similarity is merely a badly calibrated probe.
If it does not, the information is largely absent.

Protocol
--------
* Split, per range, at the *item* level (not the pair level), so no test pair
  shares a string with the training pairs.
* One probe per (model, range, split), pooled over the four quantities, since
  the four quantities share the same numeric values. Target: log1p(|v_i - v_j|).
* Repeated over REG_N_SPLITS random splits; reported values are means over
  splits, with the across-split standard deviation kept alongside.
* Evaluation: every test item serves once as anchor; Kendall's tau and pairwise
  accuracy are computed over the remaining test items and averaged.
* The identical protocol is applied to plain cosine similarity, which gives the
  directly comparable `kendall_cosine` / `pairwise_cosine` columns.

Note: unlike Table 1, the anchor is excluded from its own candidate list, so the
trivially-resolved diagonal inflates neither column. Table 4 is therefore
internally comparable but not comparable to Table 1 in absolute terms.
"""

from sklearn.linear_model import RidgeCV

REG_TEST_FRACTION = 0.35
REG_MIN_TEST = 4
REG_N_SPLITS = 5
REG_SEED = 12345
REG_ALPHAS = np.logspace(-2, 4, 13)


# --------------------------------------------------------------------
# SPLIT + FEATURES
# --------------------------------------------------------------------

def split_indices(n, test_fraction=REG_TEST_FRACTION, seed=REG_SEED):
    """Item-level split: returns (train_idx, test_idx)."""
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_test = max(REG_MIN_TEST, int(round(test_fraction * n)))
    n_test = min(n_test, n - 3)
    return np.sort(perm[n_test:]), np.sort(perm[:n_test])


def pair_features(emb, idx):
    """|e_i - e_j| for every unordered pair drawn from `idx`."""
    i, j = np.triu_indices(len(idx), k=1)
    a, b = np.asarray(idx)[i], np.asarray(idx)[j]
    return np.abs(emb[a] - emb[b]), a, b


def ranking_metrics_from_scores(score, true_dist):
    """
    score:     higher = predicted to be closer to the anchor
    true_dist: ground-truth physical distance to the anchor
    """
    score = np.asarray(score, dtype=float)
    true_dist = np.asarray(true_dist, dtype=float)

    tau = kendalltau(true_dist, -score).statistic

    correct = 0
    total = 0

    for i in range(len(score)):
        for j in range(i + 1, len(score)):
            if true_dist[i] < true_dist[j]:
                correct += score[i] > score[j]
            else:
                correct += score[i] < score[j]
            total += 1

    return tau, (correct / total if total > 0 else 0.0)


def anchored_scores(score_fn, values, test_idx):
    """Average tau / pairwise accuracy over all test items used as anchor."""
    taus, accs = [], []

    for anchor in test_idx:
        others = np.array([k for k in test_idx if k != anchor])
        true_dist = np.abs(values[others] - values[anchor])
        tau, acc = ranking_metrics_from_scores(score_fn(anchor, others), true_dist)
        taus.append(tau)
        accs.append(acc)

    return float(np.nanmean(taus)), float(np.mean(accs))


# --------------------------------------------------------------------
# PER-MODEL RUN
# --------------------------------------------------------------------

def run_regression_benchmark(model_name, encoder=None):
    """
    `encoder` is only used for testing: any object exposing
    .encode(texts, normalize_embeddings=True) works.
    """

    model_id = model_name.split("/")[-1].replace("-", "_")
    print("=" * 80)
    print(model_name)
    print("=" * 80)

    model = encoder or SentenceTransformer(model_name, trust_remote_code=True).float()

    # embed once, then drop the model to free GPU memory
    emb = {}
    for unit in units:
        for name, values in RANGES.items():
            emb[(unit, name)] = model.encode(
                range_texts(unit, name, values),
                normalize_embeddings=True,
            ).astype(np.float32)

    if encoder is None:
        del model
        gc.collect()
        torch.cuda.empty_cache()

    rows = []

    for name, raw_values in RANGES.items():

        values = np.asarray(raw_values, dtype=float)

        # split -> unit -> (tau_probe, acc_probe, tau_cos, acc_cos)
        per_split = {unit: [] for unit in units}
        alphas = []

        for s in range(REG_N_SPLITS):

            train_idx, test_idx = split_indices(len(values), seed=REG_SEED + s)

            # one probe per (range, split), pooled over the four quantities
            X, y = [], []
            for unit in units:
                feats, a, b = pair_features(emb[(unit, name)], train_idx)
                X.append(feats)
                y.append(np.log1p(np.abs(values[a] - values[b])))

            X = np.vstack(X)
            y = np.concatenate(y)

            reg = RidgeCV(alphas=REG_ALPHAS).fit(X, y)
            alphas.append(float(reg.alpha_))

            for unit in units:
                e = emb[(unit, name)]

                tau_r, acc_r = anchored_scores(
                    lambda anchor, others: -reg.predict(np.abs(e[others] - e[anchor])),
                    values,
                    test_idx,
                )
                tau_c, acc_c = anchored_scores(
                    lambda anchor, others: e[others] @ e[anchor],
                    values,
                    test_idx,
                )

                per_split[unit].append((tau_r, acc_r, tau_c, acc_c))

        for unit in units:

            arr = np.array(per_split[unit])  # (n_splits, 4)
            mean = arr.mean(axis=0)
            sd = arr.std(axis=0, ddof=1) if len(arr) > 1 else np.zeros(4)

            print(
                f"{unit:9s} {name:15s}  "
                f"probe \u03c4={mean[0]:.3f}(\u00b1{sd[0]:.3f}) pairwise={mean[1]:.3f}   |   "
                f"cosine \u03c4={mean[2]:.3f}(\u00b1{sd[2]:.3f}) pairwise={mean[3]:.3f}"
            )

            rows.append({
                "model": model_id,
                "unit": unit,
                "range": name,
                "kendall": mean[0],
                "pairwise": mean[1],
                "kendall_cosine": mean[2],
                "pairwise_cosine": mean[3],
                "kendall_sd_splits": sd[0],
                "kendall_cosine_sd_splits": sd[2],
                "alpha_median": float(np.median(alphas)),
                "n_splits": REG_N_SPLITS,
                "n_train_pairs": int(X.shape[0]),
                "n_train_items": int(len(train_idx)),
                "n_test_items": int(len(test_idx)),
            })

    df = pd.DataFrame(rows)

    summary = df.groupby("model").agg(
        kendall_mean=("kendall", "mean"),
        pairwise_mean=("pairwise", "mean"),
        kendall_std=("kendall", "std"),
        pairwise_std=("pairwise", "std"),
        kendall_cosine_mean=("kendall_cosine", "mean"),
        pairwise_cosine_mean=("pairwise_cosine", "mean"),
        kendall_cosine_std=("kendall_cosine", "std"),
        pairwise_cosine_std=("pairwise_cosine", "std"),
    ).reset_index()

    df.to_csv(
        os.path.join(CSV_DIR, f"{model_id}_benchmark_full_regression_experiment.csv"),
        index=False,
    )
    summary.to_csv(
        os.path.join(CSV_DIR, f"{model_id}_benchmark_summary_regression_experiment.csv"),
        index=False,
    )

    print()
    print(summary.to_string(index=False))

    gc.collect()
    torch.cuda.empty_cache()

    return df


def run_regression(models):
    """Run the calibration probe over a list of model names."""
    out = []
    for model_name in models:
        out.append(run_regression_benchmark(model_name))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

### Run: baseline models

In [ ]:
# Baseline models (the original paper set + Qwen3 4B / 8B)
MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "intfloat/e5-large-v2",
    "BAAI/bge-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
    "Qwen/Qwen3-Embedding-0.6B",
    "Qwen/Qwen3-Embedding-4B",
    "Qwen/Qwen3-Embedding-8B",
]

run_regression(MODELS)

sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

meter     local_0_10       probe τ=0.577(±0.167) pairwise=0.787   |   cosine τ=0.502(±0.141) pairwise=0.754
kilogram  local_0_10       probe τ=0.568(±0.190) pairwise=0.785   |   cosine τ=0.482(±0.092) pairwise=0.743
liter     local_0_10       probe τ=0.581(±0.139) pairwise=0.787   |   cosine τ=0.509(±0.131) pairwise=0.752
second    local_0_10       probe τ=0.545(±0.122) pairwise=0.773   |   cosine τ=0.393(±0.096) pairwise=0.701
meter     medium_0_1000    probe τ=0.502(±0.159) pairwise=0.745   |   cosine τ=0.398(±0.120) pairwise=0.699
kilogram  medium_0_1000    probe τ=0.421(±0.197) pairwise=0.707   |   cosine τ=0.386(±0.110) pairwise=0.697
liter     medium_0_1000    probe τ=0.452(±0.189) pairwise=0.728   |   cosine τ=0.451(±0.124) pairwise=0.730
second    medium_0_1000    probe τ=0.463(±0.176) pairwise=0.730   |   cosine τ=0.409(±0.085) pairwise=0.709
meter     log_1_100k       probe τ=0.122(±0.146) pairwise=0.560   |   cosine τ=0.233(±0.166) pairwise=0.617
kilogram  log_1_100k       p

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

meter     local_0_10       probe τ=0.601(±0.133) pairwise=0.798   |   cosine τ=0.396(±0.164) pairwise=0.703
kilogram  local_0_10       probe τ=0.601(±0.168) pairwise=0.798   |   cosine τ=0.353(±0.133) pairwise=0.684
liter     local_0_10       probe τ=0.710(±0.091) pairwise=0.848   |   cosine τ=0.485(±0.097) pairwise=0.749
second    local_0_10       probe τ=0.713(±0.147) pairwise=0.851   |   cosine τ=0.331(±0.139) pairwise=0.670
meter     medium_0_1000    probe τ=0.451(±0.142) pairwise=0.720   |   cosine τ=0.394(±0.158) pairwise=0.691
kilogram  medium_0_1000    probe τ=0.448(±0.177) pairwise=0.724   |   cosine τ=0.429(±0.171) pairwise=0.709
liter     medium_0_1000    probe τ=0.454(±0.226) pairwise=0.724   |   cosine τ=0.455(±0.199) pairwise=0.722
second    medium_0_1000    probe τ=0.444(±0.206) pairwise=0.720   |   cosine τ=0.480(±0.160) pairwise=0.733
meter     log_1_100k       probe τ=0.182(±0.076) pairwise=0.591   |   cosine τ=0.507(±0.150) pairwise=0.752
kilogram  log_1_100k       p

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

meter     local_0_10       probe τ=0.523(±0.179) pairwise=0.754   |   cosine τ=0.364(±0.094) pairwise=0.676
kilogram  local_0_10       probe τ=0.356(±0.236) pairwise=0.670   |   cosine τ=0.209(±0.091) pairwise=0.606
liter     local_0_10       probe τ=0.394(±0.253) pairwise=0.688   |   cosine τ=0.332(±0.084) pairwise=0.659
second    local_0_10       probe τ=0.540(±0.205) pairwise=0.768   |   cosine τ=0.309(±0.130) pairwise=0.653
meter     medium_0_1000    probe τ=0.481(±0.194) pairwise=0.735   |   cosine τ=0.308(±0.096) pairwise=0.650
kilogram  medium_0_1000    probe τ=0.504(±0.196) pairwise=0.750   |   cosine τ=0.231(±0.157) pairwise=0.615
liter     medium_0_1000    probe τ=0.471(±0.193) pairwise=0.737   |   cosine τ=0.297(±0.152) pairwise=0.644
second    medium_0_1000    probe τ=0.540(±0.212) pairwise=0.762   |   cosine τ=0.375(±0.258) pairwise=0.684
meter     log_1_100k       probe τ=0.478(±0.062) pairwise=0.739   |   cosine τ=0.469(±0.060) pairwise=0.735
kilogram  log_1_100k       p

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

meter     local_0_10       probe τ=0.461(±0.086) pairwise=0.722   |   cosine τ=0.307(±0.079) pairwise=0.651
kilogram  local_0_10       probe τ=0.439(±0.225) pairwise=0.712   |   cosine τ=0.267(±0.128) pairwise=0.634
liter     local_0_10       probe τ=0.441(±0.236) pairwise=0.720   |   cosine τ=0.267(±0.155) pairwise=0.632
second    local_0_10       probe τ=0.475(±0.147) pairwise=0.730   |   cosine τ=0.352(±0.112) pairwise=0.672
meter     medium_0_1000    probe τ=0.478(±0.212) pairwise=0.737   |   cosine τ=0.233(±0.092) pairwise=0.619
kilogram  medium_0_1000    probe τ=0.355(±0.231) pairwise=0.672   |   cosine τ=0.172(±0.182) pairwise=0.589
liter     medium_0_1000    probe τ=0.395(±0.166) pairwise=0.693   |   cosine τ=0.257(±0.086) pairwise=0.627
second    medium_0_1000    probe τ=0.448(±0.093) pairwise=0.722   |   cosine τ=0.419(±0.150) pairwise=0.705
meter     log_1_100k       probe τ=0.439(±0.124) pairwise=0.720   |   cosine τ=0.442(±0.087) pairwise=0.721
kilogram  log_1_100k       p

,model,unit,range,kendall,pairwise,kendall_cosine,pairwise_cosine,kendall_sd_splits,kendall_cosine_sd_splits,alpha_median,n_splits,n_train_pairs,n_train_items,n_test_items
0,all_MiniLM_L6_v2,meter,local_0_10,0.576620,0.786667,0.501545,0.754286,0.166925,0.140543,0.01,5,364,14,7
1,all_MiniLM_L6_v2,kilogram,local_0_10,0.567783,0.784762,0.481814,0.742857,0.190263,0.092168,0.01,5,364,14,7
2,all_MiniLM_L6_v2,liter,local_0_10,0.580728,0.786667,0.509030,0.752381,0.138605,0.131197,0.01,5,364,14,7
3,all_MiniLM_L6_v2,second,local_0_10,0.545342,0.773333,0.393496,0.700952,0.122473,0.095776,0.01,5,364,14,7
4,all_MiniLM_L6_v2,meter,medium_0_1000,0.502110,0.744762,0.398138,0.699048,0.159232,0.119772,0.01,5,364,14,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,Qwen3_Embedding_8B,second,sign_split,0.489269,0.744762,0.481829,0.740952,0.145291,0.137654,0.01,5,364,14,7
156,Qwen3_Embedding_8B,meter,scientific,0.426984,0.713492,0.319048,0.659524,0.096454,0.122104,0.01,5,480,16,9
157,Qwen3_Embedding_8B,kilogram,scientific,0.415873,0.707937,0.292063,0.646032,0.065108,0.120937,0.01,5,480,16,9
158,Qwen3_Embedding_8B,liter,scientific,0.441270,0.720635,0.320635,0.660317,0.123794,0.115093,0.01,5,480,16,9


### Run: extended models

#### PT1

In [ ]:
NEW_MODELS_1 = [
    "sentence-transformers/LaBSE",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "intfloat/multilingual-e5-large-instruct",
    "BAAI/bge-m3",
    "google/embeddinggemma-300m",
]

run_regression(NEW_MODELS_1)

#### PT2

In [ ]:
NEW_MODELS_2 = [
    "ibm-granite/granite-embedding-107m-multilingual",
    "ibm-granite/granite-embedding-278m-multilingual",
    "ibm-granite/granite-embedding-english-r2",
    "ibm-granite/granite-embedding-97m-multilingual-r2",
    "ibm-granite/granite-embedding-311m-multilingual-r2",
]

run_regression(NEW_MODELS_2)

#### PT3

In [ ]:
NEW_MODELS_3 = [
    "mixedbread-ai/mxbai-embed-large-v1",
    "microsoft/harrier-oss-v1-270m",
    "microsoft/harrier-oss-v1-0.6b",
    "lightonai/DenseOn",
]

run_regression(NEW_MODELS_3)

# Tables

Regenerates the LaTeX tables from the CSVs on disk, so they stay in sync with
whatever has been run. Requires the Exam 1 definitions cell (for `CSV_DIR`,
`TEX_DIR`).

* `table1_model_ranking.tex` - paper Table 1
* `table2_unit_range.tex` - paper Table 2
* `table3_lexical.tex` - paper Table 3
* `table4_regression_probe.tex` - new, Exam 3

In [ ]:
"""
LaTeX tables, regenerated from the CSVs in produced_results/csv, so they stay in
sync with whatever has been run.

  table1_model_ranking.tex      -- aggregate model ranking (paper Table 1)
  table2_unit_range.tex         -- per unit/range tau across models (Table 2)
  table3_lexical.tex            -- lexical vs. numerical association (Table 3)
  table4_regression_probe.tex   -- calibration probe vs. cosine (new)

Each function takes a `models` list, so the same code produces the main-paper
table and an appendix table over everything on disk (`models=None`).

Requires the Exam 1 definitions cell for CSV_DIR / TEX_DIR.
"""

import glob

KEY = ["model", "unit", "range"]
LEX_KEY = ["model", "group_type", "group", "reference"]

# models shown in the main paper; None = every model found on disk
MAIN_MODELS = [
    "all_MiniLM_L6_v2",
    "all_mpnet_base_v2",
    "e5_large_v2",
    "bge_large_en_v1.5",
    "nomic_embed_text_v1.5",
    "Qwen3_Embedding_0.6B",
    "Qwen3_Embedding_4B",
    "Qwen3_Embedding_8B",
]


def _load(pattern, key, models):
    files = sorted(glob.glob(os.path.join(CSV_DIR, pattern)))
    if not files:
        raise FileNotFoundError(f"no files matching {pattern} in {CSV_DIR}")

    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df = df.drop_duplicates(subset=key, keep="last")

    if models is not None:
        df = df[df["model"].isin(models)]

    return df.reset_index(drop=True)


def load_results(pattern="*_benchmark_full.csv", models=None):
    """Per-model benchmark CSVs, with the cumulative duplicates dropped."""
    return _load(pattern, KEY, models)


def load_lexical(models=None):
    """Per-model lexical CSVs (long format)."""
    return _load("*_lexical.csv", LEX_KEY, models)


def _num(v, width=5, decimals=2):
    """00.00 / -07.33 style, matching the paper."""
    if pd.isna(v):
        return "--"
    return f"-{abs(v):0{width}.{decimals}f}" if v < 0 else f"{v:0{width}.{decimals}f}"


def _num1(v):
    return _num(v, width=4, decimals=1)


def _name(m):
    return m.replace("_", r"\_")


def _order(df, models):
    """Keep the caller's model order; fall back to alphabetical."""
    if models is None:
        return sorted(df["model"].unique())
    return [m for m in models if m in set(df["model"])]


def _write(lines, filename):
    path = os.path.join(TEX_DIR, filename)
    text = "\n".join(lines) + "\n"
    with open(path, "w") as f:
        f.write(text)
    print(text)
    print(f"-> {path}\n")
    return path


# --------------------------------------------------------------------
# Table 1 -- aggregate model ranking
# --------------------------------------------------------------------

def table_model_ranking(models=MAIN_MODELS, filename="table1_model_ranking.tex"):

    df = load_results(models=models)

    agg = (df.groupby("model")[["kendall", "pairwise"]].mean() * 100)
    agg["avg"] = (agg["kendall"] + agg["pairwise"]) / 2
    agg = agg.sort_values("avg", ascending=False)

    lines = [
        r"\begin{tabular}{lcc|c}",
        r"\toprule",
        r"model & K's $\tau$ & PACC & AVG \\",
        r"\midrule",
    ]

    for model, r in agg.iterrows():
        lines.append(
            f"{_name(model)} & {_num(r['kendall'])} & {_num(r['pairwise'])} & {_num(r['avg'])} \\\\"
        )

    all_tau = df["kendall"].mean() * 100
    all_pacc = df["pairwise"].mean() * 100

    lines += [
        r"\midrule",
        f"all & {_num(all_tau)} & {_num(all_pacc)} & {_num((all_tau + all_pacc) / 2)} \\\\",
        r"random-baseline & 00.00 & 50.00 & 25.00 \\",
        r"\bottomrule",
        r"\end{tabular}",
    ]

    return _write(lines, filename)


# --------------------------------------------------------------------
# Table 2 -- average / min / max tau per unit and range
# --------------------------------------------------------------------

def table_unit_range(models=MAIN_MODELS, filename="table2_unit_range.tex"):

    df = load_results(models=models)

    agg = (df.groupby(["unit", "range"])["kendall"].agg(["mean", "min", "max"]) * 100)

    lines = [
        r"\begin{tabular}{llccc}",
        r"\toprule",
        r"Unit & Range & Avg & Min & Max \\",
        r"\midrule",
    ]

    for unit in sorted(agg.index.get_level_values("unit").unique()):
        sub = agg.loc[unit].sort_index()

        for range_name, r in sub.iterrows():
            lines.append(
                f"{unit} & {_name(range_name)} & "
                f"{_num(r['mean'])} & {_num(r['min'])} & {_num(r['max'])} \\\\"
            )

        lines.append(
            f"{unit} & AVG & "
            f"{_num(sub['mean'].mean())} & {_num(sub['min'].mean())} & {_num(sub['max'].mean())} \\\\"
        )
        lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.append(r"\end{tabular}")

    return _write(lines, filename)


# --------------------------------------------------------------------
# Table 3 -- lexical vs. numerical association
# --------------------------------------------------------------------

LEX_COLUMNS = [
    ("overall", "all", "Overall"),
    ("type_pair", "float-float", "FF"),
    ("type_pair", "float-int", "FI"),
    ("type_pair", "int-int", "II"),
]

LEX_REFERENCES = [
    ("char_lev", "Character"),
    ("token_lev", "Tokenizer"),
    ("num_dist", "Numeric"),
]


def table_lexical(models=MAIN_MODELS, filename="table3_lexical.tex"):

    df = load_lexical(models=models)

    lookup = {
        (r.model, r.group_type, r.group, r.reference): r.tau
        for r in df.itertuples()
    }

    lines = [
        r"\begin{tabular}{lcccc|cccc|cccc}",
        r"\toprule",
        r"& \multicolumn{4}{c}{Character}"
        r"& \multicolumn{4}{c}{Tokenizer}"
        r"& \multicolumn{4}{c}{Numeric} \\",
        r"\cmidrule(lr){2-5}\cmidrule(lr){6-9}\cmidrule(lr){10-13}",
        r"Model"
        r" & Overall & FF & FI & II"
        r" & Overall & FF & FI & II"
        r" & Overall & FF & FI & II \\",
        r"\midrule",
    ]

    for model in _order(df, models):
        row = [_name(model)]
        for reference, _ in LEX_REFERENCES:
            for group_type, group, _ in LEX_COLUMNS:
                tau = lookup.get((model, group_type, group, reference))
                row.append(_num1(100 * tau) if tau is not None else "--")
        lines.append(" & ".join(row) + r" \\")

    lines += [r"\bottomrule", r"\end{tabular}"]

    return _write(lines, filename)


# --------------------------------------------------------------------
# Table 4 -- calibration probe vs. cosine
# --------------------------------------------------------------------

def table_regression_probe(models=MAIN_MODELS, filename="table4_regression_probe.tex"):

    df = load_results(
        pattern="*_benchmark_full_regression_experiment.csv",
        models=models,
    )

    cols = ["kendall_cosine", "pairwise_cosine", "kendall", "pairwise"]
    agg = (df.groupby("model")[cols].mean() * 100)
    agg["delta"] = agg["kendall"] - agg["kendall_cosine"]
    agg = agg.sort_values("kendall", ascending=False)

    lines = [
        r"\begin{tabular}{lcc|cc|c}",
        r"\toprule",
        r"& \multicolumn{2}{c}{Cosine} & \multicolumn{2}{c}{Linear probe} & \\",
        r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
        r"model & K's $\tau$ & PACC & K's $\tau$ & PACC & $\Delta\tau$ \\",
        r"\midrule",
    ]

    for model, r in agg.iterrows():
        lines.append(
            f"{_name(model)} & "
            f"{_num(r['kendall_cosine'])} & {_num(r['pairwise_cosine'])} & "
            f"{_num(r['kendall'])} & {_num(r['pairwise'])} & {_num(r['delta'])} \\\\"
        )

    m = agg.mean()
    lines += [
        r"\midrule",
        f"all & {_num(m['kendall_cosine'])} & {_num(m['pairwise_cosine'])} & "
        f"{_num(m['kendall'])} & {_num(m['pairwise'])} & {_num(m['delta'])} \\\\",
        r"\bottomrule",
        r"\end{tabular}",
    ]

    return _write(lines, filename)

### Main paper tables

In [ ]:
table_model_ranking()
table_unit_range()
table_lexical()
table_regression_probe()

### Appendix tables (every model found in `produced_results/csv`)

In [ ]:
table_model_ranking(models=None, filename="appendix_table1_model_ranking.tex")
table_unit_range(models=None, filename="appendix_table2_unit_range.tex")
table_lexical(models=None, filename="appendix_table3_lexical.tex")
table_regression_probe(models=None, filename="appendix_table4_regression_probe.tex")

---
# Tokenizer scratch cells

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-Embedding-0.6B",
        trust_remote_code=True,
    )

In [ ]:
print(tokenizer.encode(["hello"], add_special_tokens=False))
print(tokenizer.encode(["0.123"], add_special_tokens=False))
print(tokenizer.encode(["11"], add_special_tokens=False))

# Export

Zips `produced_results/` (pdf + csv + tex) and downloads it. The inventory
printed first is there so you can see at a glance whether a model is missing
before you export.

In [ ]:
import os
import shutil

folder_path = "produced_results"
zip_base = "produced_results_current"

# --- inventory: confirm everything you expect is actually on disk ---
for sub in ["pdf", "csv", "tex"]:
    p = os.path.join(folder_path, sub)
    files_ = sorted(os.listdir(p)) if os.path.isdir(p) else []
    print(f"{sub:4s} {len(files_):5d} files")

n_models = len({
    f.split("_benchmark_full.csv")[0]
    for f in os.listdir(os.path.join(folder_path, "csv"))
    if f.endswith("_benchmark_full.csv")
})
print(f"\nmodels with Exam 1 results: {n_models}")

# --- zip ---
zip_path = shutil.make_archive(zip_base, "zip", folder_path)
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

from google.colab import files
files.download(zip_path)

### Fallback: copy the zip to Drive instead

In [ ]:
# Fallback: browser downloads of large zips sometimes stall in Colab.
# This copies the same zip to Google Drive instead.
from google.colab import drive

drive.mount("/content/drive")
shutil.copy(zip_path, "/content/drive/MyDrive/")
print("copied to /content/drive/MyDrive/")